# TreeMemory Real LLM Benchmark

This notebook tests TreeMemory with a real language model. The experiment asks whether an LLM answers more reliably when it receives cleaner hierarchical memory context instead of noisy flat-memory context.

## Runtime

Use a GPU runtime if available:

`Runtime -> Change runtime type -> T4 GPU`

CPU also works with small settings, but it will be slower.

In [ ]:
REPO_URL = "https://github.com/g1g4b1t/tree-memory.git"

import os
import shutil
import subprocess
from pathlib import Path

repo_dir = Path("/content/tree-memory")
if repo_dir.exists():
    shutil.rmtree(repo_dir)

subprocess.run(["git", "clone", REPO_URL, str(repo_dir)], check=True)
os.chdir(repo_dir)
print("Cloned to", Path.cwd())
print("Commit:")
subprocess.run(["git", "rev-parse", "--short", "HEAD"], check=False)

required_paths = [
    "benchmarks/llm_context_benchmark.py",
    "benchmarks/scaled_memory_benchmark.py",
    "requirements-llm.txt",
]
missing = [path for path in required_paths if not Path(path).exists()]
if missing:
    raise FileNotFoundError("Missing required files after clone: " + ", ".join(missing))
print("Required files found.")

## Install LLM Dependencies

In [ ]:
import subprocess
import sys

# Colab already provides PyTorch with the selected CPU/GPU runtime.
# Reinstalling torch can break CUDA or fail because of large downloads, so
# this cell installs only the lightweight missing packages.
import importlib.util

needed = []
for package in ["pandas", "transformers"]:
    if importlib.util.find_spec(package) is None:
        needed.append(package)

if needed:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *needed], check=True)

import torch
print("LLM dependencies ready.")
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

## Run the LLM Context Benchmark

Default quick model: `google/flan-t5-small`.

This compares:

- no context
- flat memory context
- hard tree context
- hybrid tree context

In [ ]:
from pathlib import Path

script = Path("benchmarks/llm_context_benchmark.py")
if not script.exists():
    raise FileNotFoundError(
        "benchmarks/llm_context_benchmark.py is missing. "
        "Upload it to GitHub, then restart the Colab runtime and run from the first cell."
    )

cmd = [
    sys.executable,
    str(script),
    "--model", "google/flan-t5-small",
    "--max-queries", "20",
    "--device", "auto",
    "--print-every", "8",
]

print("Running:", " ".join(cmd))
proc = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(proc.stdout)
if proc.returncode != 0:
    raise RuntimeError(f"LLM benchmark failed with exit code {proc.returncode}. Full log is printed above.")

## Inspect Results

In [ ]:
import json
import pandas as pd

with open("artifacts/results/llm_context_benchmark_results.json", "r", encoding="utf-8") as f:
    results = json.load(f)

pd.DataFrame(results["overall"])